# 1. Configuração do Ambiente

In [7]:
import os, math, warnings, json
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
from datetime import datetime
import seaborn as sns

# Estilo dos gráficos
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 7)

# Ignorar warnings
warnings.filterwarnings("ignore")

# Diretório de saída
OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Identificador da execução
RUN_ID = datetime.now().strftime("%Y%m%d-%H%M%S")

# 2. Parâmetros e Caminhos

In [8]:
# Caminhos dos datasets
TRAIN_PATH = os.path.join("..", "data", "samples", "amostra_parte_1.csv")
PROD_PATH  = os.path.join("..", "data", "samples", "amostra_tiny.csv")

# Colunas a ignorar
ID_COLS = []                    # IDs ou chaves
TARGET_COLS = []                # Targets ou rótulos

# Heurísticas
CATEG_THRESHOLD_UNIQUE = 20     # <=20 únicos → categórica
N_BINS = 20

# Limiares para detecção de drift
PSI_DRIFT = 0.25                # PSI > 0.25 → alto drift
KS_P_THRESHOLD = 0.01           # p-valor < 0.01 → diferença significativa
CHI2_P_THRESHOLD = 0.01         # idem para categóricas

# 3. Funções Utilitárias

In [9]:
def classify_col(s: pd.Series, cat_thr=20):
    """Classifica coluna como contínua ou categórica."""
    if pd.api.types.is_numeric_dtype(s):
        return "categorical" if s.dropna().nunique() <= cat_thr else "continuous"
    return "categorical"

def _safe_hist(x, bins):
    """Histograma seguro para cálculo de PSI."""
    x = x[~pd.isnull(x)]
    if len(x) == 0:
        return np.zeros(len(bins) - 1)
    h, _ = np.histogram(x, bins=bins, density=True)
    h = np.clip(h, 1e-12, None)
    return h / h.sum()

def psi(expected, actual, bins=20):
    """Calcula Population Stability Index (PSI)."""
    e = expected.astype(float)
    a = actual.astype(float)
    allv = np.concatenate([e[~np.isnan(e)], a[~np.isnan(a)]])
    if allv.size == 0:
        return np.nan

    bin_edges = np.histogram_bin_edges(allv, bins=bins)
    e_hist = _safe_hist(e, bin_edges)
    a_hist = _safe_hist(a, bin_edges)

    return np.sum((e_hist - a_hist) * np.log(e_hist / a_hist))

# 4. Carregamento e Preparação dos Dados

In [10]:
# Carregar datasets
train = pd.read_csv(TRAIN_PATH)
prod  = pd.read_csv(PROD_PATH)

# Selecionar colunas comuns, excluindo IDs e targets
ignore = set(ID_COLS + TARGET_COLS)
use_cols = [c for c in train.columns if c in prod.columns and c not in ignore]

train = train[use_cols].copy()
prod  = prod[use_cols].copy()

# Classificar tipo de cada coluna
types = {c: classify_col(train[c], CATEG_THRESHOLD_UNIQUE) for c in use_cols}

# Informações iniciais
print(f"Treino: {train.shape} | Produção: {prod.shape}")
print(f"Colunas analisadas: {len(use_cols)}")

Treino: (5000, 79) | Produção: (1000, 79)
Colunas analisadas: 79
